In [21]:
import pandas as pd
import logging
import os

os.makedirs('logs', exist_ok=True)

logging.basicConfig(
    filename='logs/data_collection.log',
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)
logger = logging.getLogger(__name__)

#loading the data
scihra = pd.read_csv('/Users/annayao/DS-4320-Project-1/data_organizer/scihra.csv')
daigt = pd.read_csv('/Users/annayao/DS-4320-Project-1/data_organizer/daigt.csv')
test = pd.read_csv('/Users/annayao/DS-4320-Project-1/data_organizer/test.csv')
train = pd.read_csv('/Users/annayao/DS-4320-Project-1/data_organizer/train.csv')

scihra.head()

,hgt,art,agt,full_text,prompt_art,prompt_agt,input_tokens_art,output_tokens_art,input_tokens_agt,output_tokens_agt,source,article_id,category
0,This study aims to examine the role of mathema...,This study explores the role of mathematical a...,This study investigates the role of mathematic...,['BETHLEHEM UNIVERSITY JOURNAL 34 (2017) Mathe...,You are an AI academic writing assistant speci...,You are an AI acting as the author of a resear...,408,210,8732,268,jstor,http://www.jstor.org/stable/10.13169/bethunivj...,Area Studies
1,The article explores how the Swedish migrants ...,This article examines how Swedish migrants nav...,This study examines how Swedish migrants on th...,['Special Issue Article • DOI: 10.1515/njmr-20...,You are an AI academic writing assistant speci...,You are an AI acting as the author of a resear...,337,137,11279,262,jstor,http://www.jstor.org/stable/48711411,Area Studies
2,"Turkish policy towards the Syrian civil war, a...","Turkish policy toward the Syrian civil war, pa...",This study examines Turkey's dynamic foreign p...,['Journal of Strategic Security Volume 11 | Nu...,You are an AI academic writing assistant speci...,You are an AI acting as the author of a resear...,385,182,15009,286,jstor,http://www.jstor.org/stable/26627190,Social Sciences
3,The debate about conservation and human welfar...,The intersection of conservation and human wel...,Community-based conservation (CBC) has been pr...,"['Conservation and Society 13(3): 244-253, 201...",You are an AI academic writing assistant speci...,You are an AI acting as the author of a resear...,382,193,11921,300,jstor,http://www.jstor.org/stable/26393203,Social Sciences
4,Several misunderstandings obscure the understa...,Misunderstandings surrounding access to pastor...,This study examines the dynamics of access to ...,"['Gonin, A., et al. (2019). Dynamics of Access...",You are an AI academic writing assistant speci...,You are an AI acting as the author of a resear...,429,208,12934,279,jstor,http://www.jstor.org/stable/26819586,Social Sciences


In [22]:
# Drop unnecessary columns from scihra
scihra.drop(columns=['article_id','output_tokens_agt',"input_tokens_agt", "art","full_text", "prompt_art","prompt_agt","input_tokens_art","output_tokens_art"], inplace=True)
scihra.head()

,hgt,agt,source,category
0,This study aims to examine the role of mathema...,This study investigates the role of mathematic...,jstor,Area Studies
1,The article explores how the Swedish migrants ...,This study examines how Swedish migrants on th...,jstor,Area Studies
2,"Turkish policy towards the Syrian civil war, a...",This study examines Turkey's dynamic foreign p...,jstor,Social Sciences
3,The debate about conservation and human welfar...,Community-based conservation (CBC) has been pr...,jstor,Social Sciences
4,Several misunderstandings obscure the understa...,This study examines the dynamics of access to ...,jstor,Social Sciences


In [23]:
# import nltk for tokenization
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/annayao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/annayao/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [24]:
import uuid
import logging
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
#schira dataset -> relationalize into 4 tables

try:
    # Ensure required NLTK tokenizer models are available
    nltk.download('punkt', quiet=True)
except Exception as e:
    logging.error(f"Failed to download NLTK dependencies: {e}")
    raise

logging.info("Building Static Tables (Authors & Model Info)...")

try:
    # Define static metadata tables for authors and models
    authors_data = [
        {'Author_ID': 'AUTH_HUMAN', 'Author_Type': 'Human', 'Description': 'Original Academic Author'},
        {'Author_ID': 'AUTH_AI_GPT', 'Author_Type': 'AI', 'Description': 'ChatGPT 4o model'}
    ]
    df_authors = pd.DataFrame(authors_data)

    model_data = [
        {'Model_ID': 'MOD_GPT', 'Author_ID': 'AUTH_AI_GPT', 'Model_Name': 'gpt'}
    ]
    df_models = pd.DataFrame(model_data)
except Exception as e:
    logging.error(f"Failed to initialize static metadata tables: {e}")
    raise

logging.info("Processing SciHRA Documents and Text Chunks...")

documents_data = []
text_chunks_data = []

try:
    # Process the source dataset to extract documents and tokenize text chunks
    for index, row in scihra.iterrows():
        try:
            # Extract shared metadata safely
            source_val = row.get('source', 'UNKNOWN')
            category_val = row.get('category', 'UNKNOWN')

            human_text = str(row['hgt']) if 'hgt' in row else ''
            
            # Process human-generated text if valid
            if pd.notna(human_text) and len(human_text.strip()) > 0:
                doc_id_human = str(uuid.uuid4())
                
                documents_data.append({
                    'Doc_ID': doc_id_human,
                    'Author_ID': 'AUTH_HUMAN',
                    'Source': source_val,
                    'Category': category_val
                })
                
                # Tokenize into sentences and generate chunk records
                sentences_human = sent_tokenize(human_text)
                for seq, sentence in enumerate(sentences_human):
                    text_chunks_data.append({
                        'Chunk_ID': str(uuid.uuid4()),
                        'Doc_ID': doc_id_human,
                        'Sequence_Number': seq,
                        'Raw_Text': sentence
                    })

            ai_text = str(row['agt']) if 'agt' in row else ''
            
            # Process AI-generated text if valid
            if pd.notna(ai_text) and len(ai_text.strip()) > 0:
                doc_id_ai = str(uuid.uuid4())
                
                documents_data.append({
                    'Doc_ID': doc_id_ai,
                    'Author_ID': 'AUTH_AI_GPT', 
                    'Source': source_val,
                    'Category': category_val
                })
                
                sentences_ai = sent_tokenize(ai_text)
                for seq, sentence in enumerate(sentences_ai):
                    text_chunks_data.append({
                        'Chunk_ID': str(uuid.uuid4()),
                        'Doc_ID': doc_id_ai,
                        'Sequence_Number': seq,
                        'Raw_Text': sentence
                    })
                    
        except Exception as row_error:
            # Log row-level errors as warnings to prevent full pipeline failure
            logging.warning(f"Error processing row {index}. Skipping. Details: {row_error}")
            continue
            
except Exception as e:
    logging.error(f"Fatal error during DataFrame iteration: {e}")
    raise

try:
    # Convert aggregated data to DataFrames
    df_documents = pd.DataFrame(documents_data)
    df_text_chunks = pd.DataFrame(text_chunks_data)

    logging.info("--- Final Row Counts ---")
    logging.info(f"Authors Table: {len(df_authors)} rows")
    logging.info(f"Models Table: {len(df_models)} rows")
    logging.info(f"Documents Table: {len(df_documents)} rows")
    logging.info(f"Text Chunks Table: {len(df_text_chunks)} rows")
except Exception as e:
    logging.error(f"Failed to consolidate processed data into DataFrames: {e}")
    raise

try:
    # Export relational tables to Parquet for optimized storage
    logging.info("Saving to Parquet format...")
    df_authors.to_parquet('data_organizer/1_authors.parquet', engine='pyarrow')
    df_models.to_parquet('data_organizer/2_model_info.parquet', engine='pyarrow')
    df_documents.to_parquet('data_organizer/3_documents.parquet', engine='pyarrow')
    df_text_chunks.to_parquet('data_organizer/4_text_chunks.parquet', engine='pyarrow')

    logging.info("SciHRA dataset relationalization complete.")
except Exception as e:
    logging.error(f"Failed to export DataFrames to Parquet. Check file permissions or disk space: {e}")
    raise

In [25]:
# Drop the unnecessary column from the daigt dataset
daigt.drop(columns=['RDizzl3_seven'], inplace=True)

In [26]:
logging.info("Dynamically generating Authors and Model Info...")

authors_dict = {}
authors_data = []
models_data = []

try:
    # Extract unique model identifiers to build static tables
    unique_models = daigt['model'].unique()

    for model_name in unique_models:
        if str(model_name).lower() == 'human':
            author_id = 'AUTH_HUMAN'
            if model_name not in authors_dict:
                authors_dict[model_name] = author_id
                authors_data.append({'Author_ID': author_id, 'Author_Type': 'Human', 'Description': 'Human Writer'})
        else:
            # Normalize AI model names for consistent ID formatting
            clean_name = str(model_name).upper().replace(" ", "_").replace("-", "_")
            author_id = f'AUTH_AI_{clean_name}'
            model_id = f'MOD_{clean_name}'
            
            if model_name not in authors_dict:
                authors_dict[model_name] = author_id
                authors_data.append({'Author_ID': author_id, 'Author_Type': 'AI', 'Description': f'{model_name} Model'})
                models_data.append({'Model_ID': model_id, 'Author_ID': author_id, 'Model_Name': model_name})

    df_authors = pd.DataFrame(authors_data)
    df_models = pd.DataFrame(models_data)
    
    logging.info(f"Created {len(df_authors)} unique Authors and {len(df_models)} unique AI Models.")
except Exception as e:
    logging.error(f"Failed to generate dynamic metadata mapping: {e}")
    raise

logging.info("2. Processing Documents and Text Chunks...")

documents_data = []
text_chunks_data = []

try:
    # Process DAIGT dataset to extract documents and tokenize text
    for index, row in daigt.iterrows():
        try:
            raw_text = str(row['text'])
            
            if pd.isna(raw_text) or len(raw_text.strip()) == 0:
                continue

            # Safely extract metadata using dictionary gets
            model_name = row.get('model', 'UNKNOWN')
            current_author_id = authors_dict.get(model_name, 'AUTH_UNKNOWN')
            
            doc_id = str(uuid.uuid4())
            
            # Map source fields to standard relational schema
            documents_data.append({
                'Doc_ID': doc_id,
                'Author_ID': current_author_id,
                'Source': row.get('source', 'UNKNOWN'),
                'Category': row.get('prompt_name', 'UNKNOWN')  
            })
            
            # Tokenize into sentences and generate individual chunk records
            sentences = sent_tokenize(raw_text)
            for seq, sentence in enumerate(sentences):
                text_chunks_data.append({
                    'Chunk_ID': str(uuid.uuid4()),
                    'Doc_ID': doc_id,
                    'Sequence_Number': seq,
                    'Raw_Text': sentence
                })
                
            # Periodic logging to track pipeline progress
            if (index + 1) % 5000 == 0:
                logging.info(f"   ... Processed {index + 1} documents ...")
                
        except Exception as row_error:
            # Log row-level errors as warnings to prevent full pipeline failure
            logging.warning(f"Error processing row {index}. Skipping. Details: {row_error}")
            continue
            
except Exception as e:
    logging.error(f"Fatal error during DAIGT DataFrame iteration: {e}")
    raise

try:
    # Aggregate processed list data into Pandas DataFrames
    df_documents = pd.DataFrame(documents_data)
    df_text_chunks = pd.DataFrame(text_chunks_data)

    logging.info("--- Final Row Counts for DAIGT ---")
    logging.info(f"Authors Table: {len(df_authors)} rows")
    logging.info(f"Models Table: {len(df_models)} rows")
    logging.info(f"Documents Table: {len(df_documents)} rows")
    logging.info(f"Text Chunks Table: {len(df_text_chunks):,} rows") 
except Exception as e:
    logging.error(f"Failed to consolidate DAIGT chunk data into DataFrames: {e}")
    raise

try:
    # Export relational tables to Parquet format for optimized I/O
    logging.info("Saving DAIGT tables to Parquet format...")
    df_authors.to_parquet('data_organizer/1_authors_daigt.parquet', engine='pyarrow')
    df_models.to_parquet('data_organizer/2_model_info_daigt.parquet', engine='pyarrow')
    df_documents.to_parquet('data_organizer/3_documents_daigt.parquet', engine='pyarrow')
    df_text_chunks.to_parquet('data_organizer/4_text_chunks_daigt.parquet', engine='pyarrow')

    logging.info("Done! DAIGT dataset fully relationalized.")
except Exception as e:
    logging.error(f"Failed to export DAIGT DataFrames to Parquet: {e}")
    raise

In [27]:
try:
    logging.info("Combining Train and Test DataFrames...")
    combined_df = pd.concat([train, test], ignore_index=True)
    logging.info(f"Total Combined Rows: {len(combined_df):,}")
except Exception as e:
    logging.error(f"Failed to concatenate train and test DataFrames. Check variables: {e}")
    raise

try:
    logging.info("Building Static Tables (Authors & Model Info)...")
    
    # Define static metadata tables for human and AI authors
    authors_data = [
        {'Author_ID': 'AUTH_HUMAN', 'Author_Type': 'Human', 'Description': 'Original Human Author'},
        {'Author_ID': 'AUTH_AI_CHATGPT3', 'Author_Type': 'AI', 'Description': 'ChatGPT 3 Model'}
    ]
    df_authors = pd.DataFrame(authors_data)

    model_data = [
        {'Model_ID': 'MOD_CHATGPT3', 'Author_ID': 'AUTH_AI_CHATGPT3', 'Model_Name': 'ChatGPT 3'}
    ]
    df_models = pd.DataFrame(model_data)
except Exception as e:
    logging.error(f"Failed to generate static metadata tables: {e}")
    raise

logging.info("Processing Documents and Text Chunks...")

documents_data = []
text_chunks_data = []

try:
    # Process combined abstract dataset to extract documents and tokenize text
    for index, row in combined_df.iterrows():
        try:
            # Safely extract text fields using dictionary gets
            abstract_text = str(row.get('abstract', ''))
            title_text = str(row.get('title', 'Unknown Title'))
            
            # Skip rows with missing, extremely short, or malformed abstracts
            if pd.isna(abstract_text) or len(abstract_text.strip()) < 10 or abstract_text.lower() == 'nan':
                continue
                
            # Determine author ID based on binary label (1 = AI, 0 = Human)
            is_ai = int(row.get('label', 0)) == 1
            current_author_id = 'AUTH_AI_CHATGPT3' if is_ai else 'AUTH_HUMAN'
            
            doc_id = str(uuid.uuid4())
            
            # Map source fields to standard relational schema
            documents_data.append({
                'Doc_ID': doc_id,
                'Author_ID': current_author_id,
                'Source': 'Ateeqq_Abstracts', 
                'Category': title_text        
            })
            
            # Tokenize abstract into sentences and generate chunk records
            sentences = sent_tokenize(abstract_text)
            for seq, sentence in enumerate(sentences):
                text_chunks_data.append({
                    'Chunk_ID': str(uuid.uuid4()),
                    'Doc_ID': doc_id,
                    'Sequence_Number': seq,
                    'Raw_Text': sentence
                })
                
            # Periodic logging to track pipeline progress
            if (index + 1) % 5000 == 0:
                logging.info(f"   ... Processed {index + 1:,} documents ...")
                
        except Exception as row_error:
            # Log row-level errors as warnings to prevent full pipeline failure
            logging.warning(f"Error processing row {index}. Skipping. Details: {row_error}")
            continue
            
except Exception as e:
    logging.error(f"Fatal error during DataFrame iteration: {e}")
    raise

try:
    # Aggregate processed list data into Pandas DataFrames
    df_documents = pd.DataFrame(documents_data)
    df_text_chunks = pd.DataFrame(text_chunks_data)

    logging.info("--- Final Row Counts ---")
    logging.info(f"Authors Table: {len(df_authors)} rows")
    logging.info(f"Models Table: {len(df_models)} rows")
    logging.info(f"Documents Table: {len(df_documents):,} rows")
    logging.info(f"Text Chunks Table: {len(df_text_chunks):,} rows")
except Exception as e:
    logging.error(f"Failed to consolidate chunk data into DataFrames: {e}")
    raise

try:
    # Export relational tables to Parquet format for optimized I/O
    logging.info("Saving to Parquet format...")
    df_authors.to_parquet('data_organizer/1_authors_ateeqq.parquet', engine='pyarrow')
    df_models.to_parquet('data_organizer/2_model_info_ateeqq.parquet', engine='pyarrow')
    df_documents.to_parquet('data_organizer/3_documents_ateeqq.parquet', engine='pyarrow')
    df_text_chunks.to_parquet('data_organizer/4_text_chunks_ateeqq.parquet', engine='pyarrow')

    logging.info("Done! Dataset fully processed and saved with ChatGPT 3 tags.")
except Exception as e:
    logging.error(f"Failed to export DataFrames to Parquet: {e}")
    raise

In [28]:
import os
import pandas as pd
import logging

# Define target directory
DATA_DIR = 'data_organizer'

logging.info("--- MERGING RELATIONAL DATABASE ---")

# --- 1. MERGING AUTHORS ---
try:
    logging.info("Merging Authors...")
    # Add the directory path to the file names
    author_files = [os.path.join(DATA_DIR, f) for f in ['1_authors.parquet', '1_authors_daigt.parquet', '1_authors_ateeqq.parquet']]
    
    # Load available Parquet files into memory
    author_dfs = [pd.read_parquet(f) for f in author_files if os.path.exists(f)]
    
    if author_dfs:
        final_authors = pd.concat(author_dfs, ignore_index=True)
        initial_authors_count = len(final_authors)
        
        # Deduplicate entities based on unique identifier
        final_authors = final_authors.drop_duplicates(subset=['Author_ID'], keep='first')
        logging.info(f"   Combined Rows: {initial_authors_count} -> Deduplicated Unique Authors: {len(final_authors)}")
        
        # Save to the data_organizer directory
        final_authors.to_parquet(os.path.join(DATA_DIR, 'FINAL_1_authors.parquet'), engine='pyarrow')
    else:
        logging.warning("   No Author files found to merge.")
except Exception as e:
    logging.error(f"Failed during Authors merge: {e}")
    raise

# --- 2. MERGING MODEL INFORMATION ---
try:
    logging.info("Merging Model Information...")
    model_files = [os.path.join(DATA_DIR, f) for f in ['2_model_info.parquet', '2_model_info_daigt.parquet', '2_model_info_ateeqq.parquet']]
    
    model_dfs = [pd.read_parquet(f) for f in model_files if os.path.exists(f)]
    
    if model_dfs:
        final_models = pd.concat(model_dfs, ignore_index=True)
        initial_models_count = len(final_models)
        
        # Deduplicate entities based on unique identifier
        final_models = final_models.drop_duplicates(subset=['Model_ID'], keep='first')
        logging.info(f"   Combined Rows: {initial_models_count} -> Deduplicated Unique Models: {len(final_models)}")
        
        final_models.to_parquet(os.path.join(DATA_DIR, 'FINAL_2_model_info.parquet'), engine='pyarrow')
    else:
        logging.warning("   No Model Info files found to merge.")
except Exception as e:
    logging.error(f"Failed during Model Info merge: {e}")
    raise

# --- 3. MERGING DOCUMENTS ---
try:
    logging.info("Merging Documents...")
    doc_files = [os.path.join(DATA_DIR, f) for f in ['3_documents.parquet', '3_documents_daigt.parquet', '3_documents_ateeqq.parquet']]
    
    doc_dfs = [pd.read_parquet(f) for f in doc_files if os.path.exists(f)]
    
    if doc_dfs:
        final_docs = pd.concat(doc_dfs, ignore_index=True)
        logging.info(f"   Total Unique Documents: {len(final_docs):,}")
        
        final_docs.to_parquet(os.path.join(DATA_DIR, 'FINAL_3_documents.parquet'), engine='pyarrow')
    else:
        logging.warning("   No Document files found to merge.")
except Exception as e:
    logging.error(f"Failed during Documents merge: {e}")
    raise

# --- 4. MERGING TEXT CHUNKS ---
try:
    logging.info("Merging Text Chunks...")
    chunk_files = [os.path.join(DATA_DIR, f) for f in ['4_text_chunks.parquet', '4_text_chunks_daigt.parquet', '4_text_chunks_ateeqq.parquet']]
    
    chunk_dfs = [pd.read_parquet(f) for f in chunk_files if os.path.exists(f)]
    
    if chunk_dfs:
        final_chunks = pd.concat(chunk_dfs, ignore_index=True)
        logging.info(f"   Total Unique Text Chunks: {len(final_chunks):,}")
        
        final_chunks.to_parquet(os.path.join(DATA_DIR, 'FINAL_4_text_chunks.parquet'), engine='pyarrow')
    else:
        logging.warning("   No Text Chunk files found to merge.")
except Exception as e:
    logging.error(f"Failed during Text Chunks merge. Check RAM usage: {e}")
    raise

# --- FINAL SYSTEM REPORT ---
try:
    logging.info("--- MERGE COMPLETE ---")
    final_files = [
        os.path.join(DATA_DIR, 'FINAL_1_authors.parquet'), 
        os.path.join(DATA_DIR, 'FINAL_2_model_info.parquet'), 
        os.path.join(DATA_DIR, 'FINAL_3_documents.parquet'), 
        os.path.join(DATA_DIR, 'FINAL_4_text_chunks.parquet')
    ]
    
    # Calculate total size of generated files
    total_size_mb = sum(os.path.getsize(f) for f in final_files if os.path.exists(f)) / (1024 * 1024)
    logging.info(f"Total Size of Final Database: {total_size_mb:.2f} MB")
except Exception as e:
    logging.error(f"Failed to calculate final database size: {e}")